In [1]:
import io
import getpass
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

pwd = getpass.getpass("PostgreSQL password: ")
url = URL.create(
    "postgresql+psycopg2",
    username="postgres",
    password=pwd,
    host="localhost",
    port=10038,
    database="malaysia_staple_prices",
)
engine = create_engine(url)

with engine.connect() as conn:
    print(conn.execute(text("SELECT current_database(), version()")).fetchone())

PostgreSQL password:  ········


('malaysia_staple_prices', 'PostgreSQL 18.1 on x86_64-windows, compiled by msvc-19.44.35221, 64-bit')


In [2]:
raw = Path("../data/raw")

item = pd.read_parquet(raw / "lookup_item.parquet")
premise = pd.read_parquet(raw / "lookup_premise.parquet")

item = item[item["item_code"] != -1]
premise = premise[premise["premise_code"] != -1].copy()
premise["premise_type"] = premise["premise_type"].str.strip()

basket = pd.read_csv("../docs/basket_items.csv")

with engine.begin() as conn:
    conn.execute(text("TRUNCATE basket_items, dim_item, dim_premise"))
    item.to_sql("dim_item", conn, if_exists="append", index=False)
    premise.to_sql("dim_premise", conn, if_exists="append", index=False)
    basket[["item_code"]].to_sql("basket_items", conn, if_exists="append", index=False)

with engine.connect() as conn:
    for t in ["dim_item", "dim_premise", "basket_items"]:
        print(t, conn.execute(text(f"SELECT COUNT(*) FROM {t}")).scalar())

dim_item 796
dim_premise 3928
basket_items 21


In [3]:
import time

expected = {
    "2026-01": 1976483, "2026-02": 1369552, "2026-03": 1547695,
    "2026-04": 1785221, "2026-05": 1717713, "2026-06": 1763560,
}

def copy_chunk(cur, df):
    buf = io.StringIO()
    df.to_csv(buf, index=False, header=False)
    buf.seek(0)
    cur.copy_expert(
        "COPY prices (price_date, premise_code, item_code, price) "
        "FROM STDIN WITH (FORMAT csv)", buf)

raw_conn = engine.raw_connection()
try:
    cur = raw_conn.cursor()
    cur.execute("TRUNCATE prices")
    raw_conn.commit()
    for label in expected:
        t0 = time.time()
        df = pd.read_parquet(raw / f"pricecatcher_{label}.parquet")
        df = df.rename(columns={"date": "price_date"})[
            ["price_date", "premise_code", "item_code", "price"]]
        n = len(df)
        for start in range(0, n, 500_000):
            copy_chunk(cur, df.iloc[start:start + 500_000])
        raw_conn.commit()
        print(label, n, "rows loaded in", round(time.time() - t0), "seconds")
        del df
finally:
    raw_conn.close()

2026-01 1976483 rows loaded in 7 seconds
2026-02 1369552 rows loaded in 5 seconds
2026-03 1547695 rows loaded in 5 seconds
2026-04 1785221 rows loaded in 6 seconds
2026-05 1717713 rows loaded in 6 seconds
2026-06 1763560 rows loaded in 6 seconds


In [4]:
with engine.connect() as conn:
    total = conn.execute(text("SELECT COUNT(*) FROM prices")).scalar()
    print("total rows:", total, "(expected 10,160,224)")
    monthly = pd.read_sql(text("""
        SELECT to_char(price_date, 'YYYY-MM') AS month,
               COUNT(*) AS n_rows,
               COUNT(DISTINCT price_date) AS days
        FROM prices GROUP BY 1 ORDER BY 1
    """), conn)
    display(monthly)

total rows: 10160224 (expected 10,160,224)


,month,n_rows,days
0,2026-01,1976483,31
1,2026-02,1369552,25
2,2026-03,1547695,25
3,2026-04,1785221,30
4,2026-05,1717713,29
5,2026-06,1763560,27
